## Power Platform Capacity — API Data Collection
Fetches tenant-level capacity entitlement and environment-level capacity consumption from the Power Platform APIs. Runs daily and appends a snapshot to the lakehouse tables for trend reporting.

In [12]:
import requests
from datetime import date
from pyspark.sql.types import *

tenant_id     = "9a5cacd0-2bef-4dd7-ac5c-7ebe1f54f495"
client_id     = "28d48667-10ad-4563-93c3-499438dafbab"
client_secret = "<SET_VIA_SECURE_CONFIGURATION>"
snapshot_date = str(date.today())

pp_token = requests.post(
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token",
    data={"client_id": client_id, "client_secret": client_secret,
          "scope": "https://api.powerplatform.com/.default", "grant_type": "client_credentials"}
).json()["access_token"]

bap_token = requests.post(
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token",
    data={"client_id": client_id, "client_secret": client_secret,
          "scope": "https://service.powerapps.com/.default", "grant_type": "client_credentials"}
).json()["access_token"]

# Tenant capacity
tenant_data = requests.get(
    "https://api.powerplatform.com/licensing/tenantCapacity?api-version=2024-10-01",
    headers={"Authorization": f"Bearer {pp_token}"}
).json()

tenant_rows = []
for group in tenant_data.get("tenantCapacities", []):
    for e in group.get("capacityEntitlements", []):
        tenant_rows.append((
            snapshot_date,
            e.get("capacityType"),
            e.get("capacitySubType"),
            float(e.get("totalCapacity", 0)),
            round(float(e.get("totalCapacity", 0)) / 1024, 2)
        ))

tenant_schema = StructType([
    StructField("SnapshotDate", StringType()),
    StructField("CapacityType", StringType()),
    StructField("CapacitySubType", StringType()),
    StructField("TenantEntitlementMB", FloatType()),
    StructField("TenantEntitlementGB", FloatType())
])
tenant_df = spark.createDataFrame(tenant_rows, tenant_schema)
print(f"✅ Tenant rows: {tenant_df.count()}")

# Environment capacity
env_data = requests.get(
    "https://api.bap.microsoft.com/providers/Microsoft.BusinessAppPlatform/scopes/admin/environments?api-version=2020-10-01&$expand=properties.capacity,properties.addons",
    headers={"Authorization": f"Bearer {bap_token}"}
).json()

env_rows = []
for env in env_data.get("value", []):
    props = env.get("properties", {})
    for cap in props.get("capacity", []):
        env_rows.append((
            snapshot_date,
            env.get("name"),
            props.get("displayName"),
            props.get("environmentSku"),
            props.get("isDefault", False),
            props.get("location"),
            cap.get("capacityType"),
            float(cap.get("actualConsumption", 0)),
            round(float(cap.get("actualConsumption", 0)) / 1024, 2),
            float(cap.get("ratedConsumption", 0)),
            cap.get("updatedOn")
        ))

env_schema = StructType([
    StructField("SnapshotDate", StringType()),
    StructField("EnvironmentId", StringType()),
    StructField("DisplayName", StringType()),
    StructField("EnvironmentSku", StringType()),
    StructField("IsDefault", BooleanType()),
    StructField("Location", StringType()),
    StructField("CapacityType", StringType()),
    StructField("ActualConsumptionMB", FloatType()),
    StructField("ActualConsumptionGB", FloatType()),
    StructField("RatedConsumptionMB", FloatType()),
    StructField("CapacityUpdatedOn", StringType())
])
env_df = spark.createDataFrame(env_rows, env_schema)
print(f"✅ Environment rows: {env_df.count()}")

StatementMeta(, f00ab0b0-3a51-4030-975d-7ec880a5cb93, 16, Finished, Available, Finished, False)

✅ Tenant rows: 28
✅ Environment rows: 2740


## Save to Lakehouse
Appends today's snapshot to the Delta tables. Uses append mode to preserve historical data for trend analysis.

In [13]:
tenant_df.write.format("delta").mode("append").option("overwriteSchema", "true").saveAsTable("TenantCapacityEntitlement")
print("✅ TenantCapacityEntitlement saved")

env_df.write.format("delta").mode("append").option("overwriteSchema", "true").saveAsTable("EnvironmentCapacityConsumption")
print("✅ EnvironmentCapacityConsumption saved")

StatementMeta(, f00ab0b0-3a51-4030-975d-7ec880a5cb93, 17, Finished, Available, Finished, False)

✅ TenantCapacityEntitlement saved
✅ EnvironmentCapacityConsumption saved


In [4]:
import requests, json
from datetime import date
from pyspark.sql.types import *

tenant_id     = "9a5cacd0-2bef-4dd7-ac5c-7ebe1f54f495"
client_id     = "28d48667-10ad-4563-93c3-499438dafbab"
client_secret = "<SET_VIA_SECURE_CONFIGURATION>"
snapshot_date = str(date.today())

bap_token = requests.post(
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token",
    data={"client_id": client_id, "client_secret": client_secret,
          "scope": "https://service.powerapps.com/.default", "grant_type": "client_credentials"}
).json()["access_token"]

env_data = requests.get(
    "https://api.bap.microsoft.com/providers/Microsoft.BusinessAppPlatform/scopes/admin/environments?api-version=2020-10-01&$expand=properties.capacity,properties.addons",
    headers={"Authorization": f"Bearer {bap_token}"}
).json()

addon_rows = []
for env in env_data.get("value", []):
    props = env.get("properties", {})
    for addon in props.get("addons", []):
        addon_rows.append((
            snapshot_date,
            env.get("name"),
            props.get("displayName"),
            props.get("environmentSku"),
            addon.get("addonType"),
            float(addon.get("allocated", 0)),
            addon.get("addonUnit")
        ))

addon_schema = StructType([
    StructField("SnapshotDate", StringType()),
    StructField("EnvironmentId", StringType()),
    StructField("DisplayName", StringType()),
    StructField("EnvironmentSku", StringType()),
    StructField("AddonType", StringType()),
    StructField("Allocated", FloatType()),
    StructField("AddonUnit", StringType())
])

addon_df = spark.createDataFrame(addon_rows, addon_schema)
print(f"✅ Addon rows: {addon_df.count()}")

addon_df.write.format("delta").mode("append").option("overwriteSchema", "true").saveAsTable("EnvironmentAddonCapacity")
print("✅ EnvironmentAddonCapacity saved")

StatementMeta(, 696da349-51df-4d18-88ed-06f13c982a9f, 8, Finished, Available, Finished, False)

✅ Addon rows: 439
✅ EnvironmentAddonCapacity saved
